In [1]:
!pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 23.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is stil

In [2]:
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor

# 1. Verileri Oku
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

# 2. Hangi sütun ne anlama geliyor tanımla
categorical_cols = ['application_year', 'graduation_year', 'department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
numeric_cols = [
    'age', 'cgpa', 'english_exam_score', 'attendance_rate', 'failed_courses_count', 'coding_score',
    'problem_solving_score', 'data_structures_score', 'sql_score', 'machine_learning_score',
    'backend_score', 'frontend_score', 'cloud_score', 'devops_score', 'project_quality_score',
    'real_client_project_count', 'internship_count', 'internship_duration_months', 'freelance_project_count',
    'hackathon_count', 'hackathon_awards', 'portfolio_score', 'github_repo_count', 'github_avg_stars',
    'open_source_contribution_count', 'linkedin_profile_score', 'cv_quality_score', 'technical_interview_score',
    'hr_interview_score', 'communication_score', 'teamwork_score', 'leadership_score', 'presentation_score',
    'certification_count', 'bootcamp_count', 'applications_sent', 'interviews_attended'
]
text_cols = ['mentor_feedback_text']

# 3. Tipleri Zorla
for col in categorical_cols:
    train_data[col] = train_data[col].astype('category')
    test_data[col] = test_data[col].astype('category')

for col in numeric_cols:
    train_data[col] = pd.to_numeric(train_data[col], errors='coerce')
    test_data[col] = pd.to_numeric(test_data[col], errors='coerce')

for col in text_cols:
    train_data[col] = train_data[col].astype(str)
    test_data[col] = test_data[col].astype(str)

# 4. AutoGluon Formatına Çevir ve Hedef Değişkeni Belirle
train_data = TabularDataset(train_data)
test_data = TabularDataset(test_data)

# Sadece eğitim verisinden ID'yi düşür (Testte lazım olacak)
train_data = train_data.drop(columns=['student_id'])
label = 'career_success_score'

In [3]:
# Zaman sınırını saniye cinsinden belirliyoruz (Örn: 3600 = 1 Saat, 7200 = 2 Saat)
# Ne kadar uzun süre verirsen, o kadar çok model kombinasyonu dener.
time_limit_seconds = 3600

predictor = TabularPredictor(
    label=label,
    eval_metric='root_mean_squared_error' # Yarışmanın başarı kriterine göre bunu 'mse' veya 'r2' yapabilirsin
).fit(
    train_data=train_data,
    presets='best_quality',
    time_limit=time_limit_seconds,
    num_gpus=1  # Colab'daki GPU'yu kullanmasını zorluyoruz
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260613_223855"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 39.49/39.49 GB
Total GPU Memory:   Free: 39.49 GB, Allocated: 0.00 GB, Total: 39.49 GB
GPU Count:          1
Memory Avail:       81.04 GB / 83.47 GB (97.1%)
Disk Space Avail:   58.34 GB / 112.64 GB (51.8%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=Tr

In [ ]:
# Liderlik tablosu: Hangi algoritmaların tek tek ne kadar iyi performans gösterdiğini listeler
predictor.leaderboard(train_data)

# Özellik Önemi (Feature Importance): Hangi sütunların tahmine ne kadar katkı sağladığını hesaplar
feature_importance = predictor.feature_importance(train_data)
display(feature_importance)

Computing feature importance via permutation shuffling for 45 features using 5000 rows with 5 shuffle sets...
	3929.44s	= Expected runtime (785.89s per shuffle set)


In [5]:
# İçine train_data KOYMUYORUZ!
leaderboard_df = predictor.leaderboard()
display(leaderboard_df)

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L3,-8.749655,root_mean_squared_error,43.702515,2168.439713,0.000559,0.036385,3,True,21
1,WeightedEnsemble_L2,-8.768181,root_mean_squared_error,3.721605,731.953965,0.000628,0.028696,2,True,13
2,CatBoost_BAG_L2,-8.802544,root_mean_squared_error,30.767509,1792.332730,1.000825,64.094888,2,True,17
3,LightGBMXT_BAG_L2,-8.834181,root_mean_squared_error,29.987591,1761.565217,0.220907,33.327375,2,True,14
4,LightGBM_BAG_L2,-8.843385,root_mean_squared_error,29.980403,1762.058212,0.213719,33.820370,2,True,15
5,RandomForestMSE_BAG_L2,-8.844279,root_mean_squared_error,42.095678,1998.881020,12.328994,270.643178,2,True,16
6,XGBoost_BAG_L2,-8.856228,root_mean_squared_error,30.522746,1790.314930,0.756063,62.077088,2,True,20
7,ExtraTreesMSE_BAG_L2,-8.865842,root_mean_squared_error,42.202458,2034.762715,12.435775,306.524873,2,True,18
8,CatBoost_r177_BAG_L1,-8.947408,root_mean_squared_error,0.967304,116.505059,0.967304,116.505059,1,True,10
9,LightGBM_r131_BAG_L1,-8.969899,root_mean_squared_error,0.593387,64.993557,0.593387,64.993557,1,True,12


In [4]:
# Test verisi üzerinden tahmin yap
predictions = predictor.predict(test_data)

# Kaggle formatında bir DataFrame oluştur
submission_df = pd.DataFrame({
    'student_id': test_data['student_id'],
    'career_success_score': predictions
})

# CSV olarak dışa aktar (Colab sol panelindeki klasör ikonunda belirecek, oradan indirebilirsin)
submission_df.to_csv('submission_autogluon_a100_ilk.csv', index=False)
print("Submission dosyası başarıyla oluşturuldu!")

Submission dosyası başarıyla oluşturuldu!


In [ ]:
# Sıkıştırma işlemi
!zip -r autogluon_ilk_özelliksiz_hamveri_a100.zip /content/AutogluonModels

# İndirme işlemi
from google.colab import files
files.download('autogluon_ilk_özelliksiz_hamveri_a100.zip')

## Conclusion

In this quickstart tutorial we saw AutoGluon's basic fit and predict functionality using `TabularDataset` and `TabularPredictor`. AutoGluon simplifies the model training process by not requiring feature engineering or model hyperparameter tuning. Next, we recommend checking out the [Essentials Tutorial](tabular-essentials.ipynb) to learn about `presets` to use for production and competition usage. You can also check out the [in-depth tutorials](index.html) to learn more about AutoGluon's other features like customizing the training and prediction steps or extending AutoGluon with custom feature generators, models, or metrics.